# **Classification: Gradient Boosting Classifier (GBC)**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
The Gradient Boosting Classifier is a powerful ensemble of Decision Trees. Since each split in these base trees is determined by finding a discrete threshold for a single feature, the algorithm is mathematically invariant to the scale of the inputs. Whether the data is standardized, normalized, or left in its raw form, the resulting decision boundaries remain completely identical. Consequently, we will use the **Original, Unscaled Data** to preserve computational efficiency and clinical interpretability.

### **The Boosting Philosophy: Why We Do Not Use the Decision Tree Champion**
Unlike Bagging (which relies on deep, complex trees to capture variance), Gradient Boosting builds trees *sequentially*. Each new tree attempts to minimize the residual errors of the previous ensemble using a gradient-descent approach. 
Because of this sequential learning, GBC requires **Weak Learners** (shallow trees, typically with a `max_depth` of 3 to 5). If we were to inject our optimized Decision Tree champion (`max_depth=10`), the very first tree would severely overfit, leaving subsequent trees to merely amplify noise. Therefore, we let the algorithm build its own shallow trees from scratch.


## **Experiment Design**

We defined a tournament of 3 optimization levels to identify the most robust configuration. Because Gradient Boosting is highly prone to overfitting if the `learning_rate` is too high or the trees get too deep, we strictly log **both Train and Test metrics** across all runs to visually monitor the learning gap:

* **Baseline**: Utilizing Scikit-Learn's default parameters (e.g., `n_estimators=100`, `max_depth=3`, `learning_rate=0.1`) to establish a pure, unconstrained performance reference.
* **GridSearchCV**: A systematic 3-fold cross-validated search testing combinations of ensemble size (`n_estimators`), step size (`learning_rate`), tree complexity (`max_depth`), and data starvation (`subsample`).
* **Optuna Optimization**: Bayesian optimization to explore a fine-grained, continuous range of these critical hyperparameters, aiming to surgically maximize generalizable Recall without allowing the Training Accuracy to spike to 1.0.

In [ ]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score

# MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_GradientBoosting")

# Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

# Split data (80/20) maintaining class proportion
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

def log_classification_metrics(model, X_tr, y_tr, X_te, y_te, duration):
    """Logs both Train and Test metrics to explicitly monitor the Overfitting Gap"""
    y_tr_pred = model.predict(X_tr)
    y_te_pred = model.predict(X_te)
    
    # Train Partition Metrics
    mlflow.log_metric("recall_train", recall_score(y_tr, y_tr_pred))
    mlflow.log_metric("accuracy_train", accuracy_score(y_tr, y_tr_pred))
    mlflow.log_metric("f1_train", f1_score(y_tr, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("recall_test", recall_score(y_te, y_te_pred))
    mlflow.log_metric("accuracy_test", accuracy_score(y_te, y_te_pred))
    mlflow.log_metric("f1_test", f1_score(y_te, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE 
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBC_Baseline_Defaults"):
    # Calling the classifier with absolute defaults (max_depth=3, learning_rate=0.1)
    gb_base = GradientBoostingClassifier(random_state=42)
    
    start_time = time.time()
    gb_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    mlflow.log_params(gb_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_classification_metrics(gb_base, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="GBC_GridSearch"):
    # Grid testing specific values for estimators, rate, depth and subsample
    param_grid = {
        'n_estimators': [50, 100, 150, 200, 300],
        'learning_rate': [0.01, 0.1, 0.3],
        'max_depth': [2, 4, 6, 8, 10], # Watch out: depth 8 and 10 might overfit!
        'subsample': [0.7, 0.85, 1.0]
    }
    
    grid = GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train, y_train)
    duration = time.time() - start_time
    
    best_gbc_grid = grid.best_estimator_
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_classification_metrics(best_gbc_grid, X_train, y_train, X_test, y_test, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0)
    }
    
    model = GradientBoostingClassifier(random_state=42, **params)
    # Cross-validation focusing entirely on the Train set to avoid leakage
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="GBC_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=12) 
    duration = time.time() - start_time
    
    # Train final champion model
    best_gbc_opt = GradientBoostingClassifier(**study.best_params, random_state=42)
    best_gbc_opt.fit(X_train, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_classification_metrics(best_gbc_opt, X_train, y_train, X_test, y_test, duration)

print("Gradient Boosting 3-run tournament successfully completed.")

## Runs Summary

| Run | n_estimators | learning_rate | max_depth | subsample | Accuracy | F1 | Recall | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| GBC_Baseline_Defaults | 100 | 0.1 | 3 | 1.0 | 0.91985 | 0.9284343051 | 0.86650 | 28.39s |
| GBC_GridSearch | 50 | 0.3 | 10 | 0.7 | 0.90440 | 0.9164700743 | 0.87408 | 4503.13s |
| GBC_Optuna | 130 | 0.1983281732 | 7 | 0.8766413523 | 0.91645 | 0.9258815702 | 0.86975 | 931.36s |

### Additional logged parameters
- `random_state = 42` where set in code
- Hyperparameters optimized: `n_estimators`, `learning_rate`, `max_depth`, `subsample`

## Best Run Justification for Streamlit

The best run for Streamlit is **GBC_Baseline_Defaults**. Across the three executions, it offers the best balance between metrics and computational cost: it delivers the **highest Accuracy**, the **highest F1**, and a sufficiently competitive Recall, without penalizing training time.

The **GBC_GridSearch** achieved the highest Recall, but that came with a drop in Accuracy and F1 and a very high training cost, which does not make sense for a Streamlit application. The **GBC_Optuna** run slightly improved F1 compared with GridSearch, but it still stayed below the baseline in Accuracy and Recall while keeping a training time much higher than the baseline.

Thus, for a stable diabetes prediction in Streamlit, the most balanced choice is **GBC_Baseline_Defaults**:
- **Accuracy**: 0.91985
- **F1**: 0.92843
- **Recall**: 0.86650
- **Fit time**: 28.39s
